#HDBSCAN CLUSTERING

In [ ]:
!pip install -q hdbscan

import os
import numpy as np
import pandas as pd

from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_score
from hdbscan.validity import validity_index

In [ ]:
folder = "/content/drive/MyDrive/puskesmas-rute"
os.makedirs(folder, exist_ok=True)

In [ ]:
# Baca hasil Dijkstra
df_titik = pd.read_csv(
    os.path.join(folder, "lokasi_osm_nodes.csv")
)

matrix_length = np.load(
    os.path.join(folder, "matrix_length.npy")
)

if matrix_length.shape != (len(df_titik), len(df_titik)):
    raise ValueError("Ukuran matriks dan jumlah lokasi tidak sama.")

if not np.isfinite(matrix_length).all():
    raise ValueError("Masih ada nilai inf atau NaN pada matriks jarak.")

if not np.allclose(np.diag(matrix_length), 0):
    raise ValueError("Diagonal matriks jarak harus nol.")

# Pisahkan Dinkes dari data clustering
is_depot = df_titik["Input Nama"].str.contains(
    "Dinas Kesehatan|Dinkes",
    case=False,
    na=False
)

if is_depot.sum() != 1:
    raise ValueError("Periksa baris Dinkes pada data lokasi.")

idx = np.flatnonzero(~is_depot.to_numpy())

df_puskesmas = df_titik.iloc[idx].copy()
df_puskesmas = df_puskesmas.reset_index(drop=True)

# Matriks jarak antarpuskesmas
D_arah = matrix_length[np.ix_(idx, idx)].astype(float)

# Rata-rata jarak pergi dan pulang
D_master = (D_arah + D_arah.T) / 2
np.fill_diagonal(D_master, 0)

if not np.isfinite(D_master).all():
    raise ValueError("Ada jarak yang tidak valid.")

if (D_master < 0).any():
    raise ValueError("Ada jarak negatif.")

if not np.allclose(D_master, D_master.T):
    raise ValueError("Matriks belum simetris.")

if np.any(np.triu(D_master == 0, k=1)):
    raise ValueError("Ada dua lokasi berbeda dengan jarak nol.")

# Simpan matriks dan urutan data puskesmas
np.save(
    os.path.join(folder, "matrix_hdbscan.npy"),
    D_master
)

df_puskesmas.to_csv(
    os.path.join(folder, "puskesmas_hdbscan.csv"),
    index=False
)

# Matriks utama tidak boleh berubah saat fitting
D_master.setflags(write=False)

print("Jumlah puskesmas:", len(df_puskesmas))
print("Ukuran matriks:", D_master.shape)
print("Diagonal maksimum:", np.max(np.abs(np.diag(D_master))))

Jumlah puskesmas: 63
Ukuran matriks: (63, 63)
Diagonal maksimum: 0.0


In [ ]:
def evaluasi_cluster(labels):

    labels = np.asarray(labels, dtype=int)

    anggota = labels != -1
    labels_anggota = labels[anggota]

    n_cluster = len(np.unique(labels_anggota))
    n_noise = int((~anggota).sum())

    if n_cluster:
        cluster_terbesar = int(
            pd.Series(labels_anggota).value_counts().max()
        )
    else:
        cluster_terbesar = 0

    hasil = {
        "Jumlah Cluster": n_cluster,
        "Noise": n_noise,
        "Noise (%)": round(100 * n_noise / len(labels), 2),
        "Cluster Terbesar": cluster_terbesar,
        "DBCV": np.nan,
        "Silhouette": np.nan,
        "Error DBCV": "",
        "Error Silhouette": ""
    }

    if n_cluster < 2:
        return hasil

    # Silhouette untuk anggota cluster saja
    try:
        D_anggota = D_master[
            np.ix_(anggota, anggota)
        ].copy()

        np.fill_diagonal(D_anggota, 0)

        if n_cluster < len(labels_anggota):
            hasil["Silhouette"] = float(
                silhouette_score(
                    D_anggota,
                    labels_anggota,
                    metric="precomputed"
                )
            )

    except Exception as e:
        hasil["Error Silhouette"] = str(e)

    # DBCV menggunakan seluruh data termasuk noise
    try:
        hasil["DBCV"] = float(
            validity_index(
                D_master.copy(),
                labels.astype(np.int64),
                metric="precomputed",
                d=2
            )
        )

    except Exception as e:
        hasil["Error DBCV"] = str(e)

    return hasil

In [ ]:
model_awal = HDBSCAN(
    min_cluster_size=3,
    min_samples=2,
    cluster_selection_method="eom",
    metric="precomputed",
    algorithm="brute",
    copy=True
)

label_awal = model_awal.fit_predict(D_master.copy())

nilai_awal = evaluasi_cluster(label_awal)

print("Hasil parameter awal:")
display(pd.DataFrame([nilai_awal]).round(4))

df_awal = df_puskesmas.copy()
df_awal["cluster"] = label_awal

df_awal.to_csv(
    os.path.join(folder, "hasil_hdbscan_awal.csv"),
    index=False
)

Hasil parameter awal:


,Jumlah Cluster,Noise,Noise (%),Cluster Terbesar,DBCV,Silhouette,Error DBCV,Error Silhouette
0,11,17,26.98,6,0.3092,0.4033,,


In [ ]:
min_cluster_size_list = [3, 4, 5, 6, 7, 8, 10]
min_samples_list = [1, 2, 3, 4, 5]
method_list = ["eom", "leaf"]

kombinasi = [
    (mcs, ms, metode)
    for mcs, ms, metode in product(
        min_cluster_size_list,
        min_samples_list,
        method_list
    )
    if ms <= mcs
]

hasil = []

for no, (mcs, ms, metode) in enumerate(kombinasi, start=1):

    baris = {
        "min_cluster_size": mcs,
        "min_samples": ms,
        "method": metode
    }

    try:
        model = HDBSCAN(
            min_cluster_size=mcs,
            min_samples=ms,
            cluster_selection_method=metode,
            metric="precomputed",
            algorithm="brute",
            copy=True
        )

        label = model.fit_predict(D_master.copy())

        baris.update(evaluasi_cluster(label))
        baris["Status"] = "selesai"
        baris["Error Fitting"] = ""

    except Exception as e:
        baris["Status"] = "gagal fitting"
        baris["Error Fitting"] = str(e)

    hasil.append(baris)

    print(
        f"Proses tuning: {no}/{len(kombinasi)}",
        end="\r"
    )

df_tuning = pd.DataFrame(hasil)

for kolom in ["DBCV", "Silhouette"]:
    df_tuning[kolom] = pd.to_numeric(
        df_tuning[kolom],
        errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

# Simpan semua hasil tuning
df_tuning.to_csv(
    os.path.join(folder, "hasil_tuning_hdbscan.csv"),
    index=False
)

# Ambil konfigurasi dengan metrik lengkap
df_valid = df_tuning.loc[
    (df_tuning["Jumlah Cluster"] >= 2) &
    df_tuning["DBCV"].notna() &
    df_tuning["Silhouette"].notna()
].copy()

df_valid = df_valid.sort_values(
    ["DBCV", "Silhouette", "Noise (%)"],
    ascending=[False, False, True]
).reset_index(drop=True)

print("\nTotal kombinasi:", len(kombinasi))
print("Berhasil fitting:",
      (df_tuning["Status"] == "selesai").sum())
print("Dapat dievaluasi:", len(df_valid))
print("Gagal fitting:",
      (df_tuning["Status"] == "gagal fitting").sum())

print("Diagonal setelah tuning:",
      np.max(np.abs(np.diag(D_master))))

display(df_valid.head(15).round(4))

# Tampilkan error jika ada
masalah = df_tuning[
    (df_tuning["Status"] != "selesai") |
    (df_tuning["Error DBCV"].fillna("") != "") |
    (df_tuning["Error Silhouette"].fillna("") != "")
]

if not masalah.empty:
    display(masalah.head(20))

Proses tuning: 64/64
Total kombinasi: 64
Berhasil fitting: 64
Dapat dievaluasi: 62
Gagal fitting: 0
Diagonal setelah tuning: 0.0


,min_cluster_size,min_samples,method,Jumlah Cluster,Noise,Noise (%),Cluster Terbesar,DBCV,Silhouette,Error DBCV,Error Silhouette,Status,Error Fitting
0,3,1,eom,11,17,26.98,6,0.3092,0.4033,,,selesai,
1,3,1,leaf,11,17,26.98,6,0.3092,0.4033,,,selesai,
2,3,2,eom,11,17,26.98,6,0.3092,0.4033,,,selesai,
3,3,2,leaf,11,17,26.98,6,0.3092,0.4033,,,selesai,
4,3,3,eom,8,27,42.86,8,0.2040,0.4741,,,selesai,
5,3,3,leaf,8,27,42.86,8,0.2040,0.4741,,,selesai,
6,4,4,eom,3,32,50.79,15,0.1994,0.4705,,,selesai,
7,4,4,leaf,3,32,50.79,15,0.1994,0.4705,,,selesai,
8,5,3,leaf,4,24,38.10,13,0.1887,0.4326,,,selesai,
9,5,4,eom,2,36,57.14,15,0.1736,0.4627,,,selesai,


In [ ]:
# Batas awal untuk kebutuhan routing
batas_noise = 25
batas_cluster = 15

if df_valid.empty:
    raise ValueError(
        "Belum ada konfigurasi dengan DBCV dan Silhouette valid."
    )

calon = df_valid[
    (df_valid["Noise (%)"] <= batas_noise) &
    (df_valid["Cluster Terbesar"] <= batas_cluster)
].copy()

if calon.empty:
    print("Tidak ada parameter yang memenuhi batas awal.")
    print("Sementara digunakan konfigurasi DBCV tertinggi.")
    print("Jumlah noise tetap perlu diperiksa sebelum RL.")

    calon = df_valid.copy()

terbaik = calon.iloc[0]

mcs = int(terbaik["min_cluster_size"])
ms = int(terbaik["min_samples"])
metode = str(terbaik["method"])

model_final = HDBSCAN(
    min_cluster_size=mcs,
    min_samples=ms,
    cluster_selection_method=metode,
    metric="precomputed",
    algorithm="brute",
    copy=True
)

label_final = model_final.fit_predict(D_master.copy())

nilai_final = evaluasi_cluster(label_final)

df_final = df_puskesmas.copy()
df_final["cluster"] = label_final

# Simpan hasil clustering
df_final.to_csv(
    os.path.join(folder, "hasil_hdbscan_final.csv"),
    index=False
)

# Simpan puskesmas noise
df_final[df_final["cluster"] == -1].to_csv(
    os.path.join(folder, "puskesmas_noise.csv"),
    index=False
)

# Simpan parameter terpilih
pd.DataFrame([{
    "min_cluster_size": mcs,
    "min_samples": ms,
    "method": metode,
    **nilai_final
}]).to_csv(
    os.path.join(folder, "parameter_hdbscan_final.csv"),
    index=False
)

print("Parameter terpilih:", mcs, ms, metode)
print("Jumlah cluster:", nilai_final["Jumlah Cluster"])
print("Noise:", nilai_final["Noise"])
print("DBCV:", nilai_final["DBCV"])
print("Silhouette:", nilai_final["Silhouette"])

print("\nAnggota setiap cluster:")
display(df_final["cluster"].value_counts().sort_index())

print("\nDiagonal master:",
      np.max(np.abs(np.diag(D_master))))

Tidak ada parameter yang memenuhi batas awal.
Sementara digunakan konfigurasi DBCV tertinggi.
Jumlah noise tetap perlu diperiksa sebelum RL.
Parameter terpilih: 3 1 eom
Jumlah cluster: 11
Noise: 17
DBCV: 0.30916113764993797
Silhouette: 0.403318843750494

Anggota setiap cluster:


,count
cluster,
-1,17
0,3
1,3
2,5
3,4
4,3
5,4
6,5
7,6



Diagonal master: 0.0


In [3]:
import os
import numpy as np
import pandas as pd

folder = "/content/drive/MyDrive/puskesmas-rute"

df_rute = pd.read_csv(
    os.path.join(folder, "hasil_hdbscan_final.csv")
)

df_urutan = pd.read_csv(
    os.path.join(folder, "puskesmas_hdbscan.csv")
)

D = np.load(
    os.path.join(folder, "matrix_hdbscan.npy")
)

# Pastikan urutan puskesmas sesuai dengan matriks
if not df_rute["Input Nama"].equals(df_urutan["Input Nama"]):
    raise ValueError("Urutan nama puskesmas tidak sesuai matriks.")

if D.shape != (len(df_rute), len(df_rute)):
    raise ValueError("Ukuran matriks tidak sesuai jumlah lokasi.")

# Simpan label HDBSCAN asli
df_rute["cluster_hdbscan"] = df_rute["cluster"].astype(int)

# Kolom kelompok untuk distribusi
df_rute["kelompok_rute"] = df_rute["cluster_hdbscan"]

cluster_awal = sorted(
    df_rute.loc[
        df_rute["cluster_hdbscan"] != -1,
        "cluster_hdbscan"
    ].unique()
)

noise_idx = np.flatnonzero(
    df_rute["cluster_hdbscan"].to_numpy() == -1
)

# Anggota asli masing-masing cluster
anggota_awal = {
    c: np.flatnonzero(
        df_rute["cluster_hdbscan"].to_numpy() == c
    )
    for c in cluster_awal
}

# Batas percobaan ukuran kelompok
maks_anggota = 9

# Dahulukan noise yang paling dekat dengan suatu cluster
urutan_noise = sorted(
    noise_idx,
    key=lambda i: min(
        D[i, anggota_awal[c]].mean()
        for c in cluster_awal
    )
)

df_rute["jarak_penugasan_m"] = np.nan

for i in urutan_noise:

    pilihan = [
        c for c in cluster_awal
        if (df_rute["kelompok_rute"] == c).sum() < maks_anggota
    ]

    if not pilihan:
        raise ValueError("Semua kelompok sudah mencapai batas anggota.")

    cluster_tujuan = min(
        pilihan,
        key=lambda c: D[i, anggota_awal[c]].mean()
    )

    jarak = D[i, anggota_awal[cluster_tujuan]].mean()

    df_rute.loc[i, "kelompok_rute"] = cluster_tujuan
    df_rute.loc[i, "jarak_penugasan_m"] = jarak

# Periksa hasil
print("Noise awal:", len(noise_idx))
print("Jumlah kelompok:", len(cluster_awal))
print("Belum mendapatkan kelompok:",
      (df_rute["kelompok_rute"] == -1).sum())

print("\nJumlah anggota tiap kelompok:")
display(
    df_rute["kelompok_rute"]
    .value_counts()
    .sort_index()
)

print("\nPenugasan puskesmas noise:")
display(
    df_rute.loc[
        df_rute["cluster_hdbscan"] == -1,
        [
            "Input Nama",
            "cluster_hdbscan",
            "kelompok_rute",
            "jarak_penugasan_m"
        ]
    ].sort_values("jarak_penugasan_m", ascending=False)
)

df_rute.to_csv(
    os.path.join(folder, "kelompok_distribusi.csv"),
    index=False
)

Noise awal: 17
Jumlah kelompok: 11
Belum mendapatkan kelompok: 0

Jumlah anggota tiap kelompok:


,count
kelompok_rute,
0,3
1,5
2,5
3,8
4,8
5,4
6,5
7,6
8,9



Penugasan puskesmas noise:


,Input Nama,cluster_hdbscan,kelompok_rute,jarak_penugasan_m
11,Puskesmas Benowo,-1,1,8428.949591
26,Puskesmas Medokan Ayu,-1,3,7867.243535
21,Puskesmas Gunung Anyar,-1,3,7818.260060
50,Puskesmas Gayungan,-1,4,6125.443240
17,Puskesmas Sememi,-1,1,5712.154138
36,Puskesmas Kenjeran,-1,10,5659.528385
60,Puskesmas Siwalankerto,-1,4,5563.344312
53,Puskesmas Kebonsari,-1,4,5545.803533
24,Puskesmas Keputih,-1,3,5353.919182
23,Puskesmas Kalirungkut,-1,3,5197.013605


In [4]:
cek_noise = []

for i in noise_idx:

    # Jarak rata-rata ke setiap cluster asli
    jarak_cluster = {}

    for c in cluster_awal:
        jarak_cluster[c] = D[i, anggota_awal[c]].mean()

    # Cluster yang benar-benar paling dekat
    terdekat = min(jarak_cluster, key=jarak_cluster.get)

    # Cluster yang dipilih kode sebelumnya
    dipilih = int(df_rute.loc[i, "kelompok_rute"])

    cek_noise.append({
        "puskesmas": df_rute.loc[i, "Input Nama"],
        "cluster_terdekat": terdekat,
        "cluster_dipilih": dipilih,
        "jarak_terdekat_km": round(jarak_cluster[terdekat] / 1000, 2),
        "jarak_dipilih_km": round(jarak_cluster[dipilih] / 1000, 2),
        "selisih_km": round(
            (jarak_cluster[dipilih] - jarak_cluster[terdekat]) / 1000, 2
        )
    })

df_cek = pd.DataFrame(cek_noise)

df_cek = df_cek.sort_values(
    "selisih_km",
    ascending=False
)

display(df_cek)

print(
    "Noise yang tidak masuk cluster terdekat:",
    (df_cek["cluster_terdekat"] != df_cek["cluster_dipilih"]).sum()
)

,puskesmas,cluster_terdekat,cluster_dipilih,jarak_terdekat_km,jarak_dipilih_km,selisih_km
0,Puskesmas Peneleh,8,8,3.73,3.73,0.0
1,Puskesmas Benowo,1,1,8.43,8.43,0.0
2,Puskesmas Sememi,1,1,5.71,5.71,0.0
3,Puskesmas Gunung Anyar,3,3,7.82,7.82,0.0
4,Puskesmas Kalijudan,8,8,3.41,3.41,0.0
5,Puskesmas Kalirungkut,3,3,5.20,5.20,0.0
6,Puskesmas Keputih,3,3,5.35,5.35,0.0
7,Puskesmas Medokan Ayu,3,3,7.87,7.87,0.0
8,Puskesmas Mojo,8,8,4.34,4.34,0.0
9,Puskesmas Mulyorejo,8,8,4.56,4.56,0.0


Noise yang tidak masuk cluster terdekat: 0


In [1]:
import os
import numpy as np
import pandas as pd

folder = "/content/drive/MyDrive/puskesmas-rute"

df_kelompok = pd.read_csv(
    os.path.join(folder, "kelompok_distribusi.csv")
)

df_titik = pd.read_csv(
    os.path.join(folder, "lokasi_osm_nodes.csv")
)

matrix_time = np.load(
    os.path.join(folder, "matrix_time.npy")
)

# Cek apakah seluruh puskesmas sudah punya kelompok
print("Jumlah puskesmas:", len(df_kelompok))
print("Jumlah kelompok:", df_kelompok["kelompok_rute"].nunique())
print("Noise awal:", (df_kelompok["cluster_hdbscan"] == -1).sum())
print("Belum terkelompok:", (df_kelompok["kelompok_rute"] == -1).sum())

# Jumlah anggota per kelompok
ringkasan = (
    df_kelompok.groupby("kelompok_rute")
    .agg(
        jumlah_puskesmas=("Input Nama", "count"),
        noise_ditugaskan=(
            "cluster_hdbscan",
            lambda x: (x == -1).sum()
        )
    )
    .reset_index()
)

display(ringkasan)

# Validasi sebelum routing
assert len(df_kelompok) == 63
assert df_kelompok["Input Nama"].is_unique
assert df_kelompok["kelompok_rute"].notna().all()
assert (df_kelompok["kelompok_rute"] != -1).all()

# Semua puskesmas harus ada di matriks Dijkstra
assert set(df_kelompok["Input Nama"]) == set(
    df_titik.loc[
        ~df_titik["Input Nama"].str.contains(
            "Dinas Kesehatan|Dinkes",
            case=False, na=False
        ),
        "Input Nama"
    ]
)

assert matrix_time.shape == (len(df_titik), len(df_titik))
assert np.isfinite(matrix_time).all()

print("Validasi kelompok dan matriks waktu berhasil.")

Jumlah puskesmas: 63
Jumlah kelompok: 11
Noise awal: 17
Belum terkelompok: 0


,kelompok_rute,jumlah_puskesmas,noise_ditugaskan
0,0,3,0
1,1,5,2
2,2,5,0
3,3,8,4
4,4,8,5
5,5,4,0
6,6,5,0
7,7,6,0
8,8,9,4
9,9,5,0


Validasi kelompok dan matriks waktu berhasil.
